In [18]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os

sns.set(style="whitegrid")
#--------
file_path = r'C:\Users\laksh\Documents\ecommerce-funnel-analysis\data\ecommerce_funnel.csv'
df = pd.read_csv(file_path)

#----------
df['event_time'] = pd.to_datetime(df['event_time'], dayfirst=True)
df['date'] = df['event_time'].dt.date
df['hour'] = df['event_time'].dt.hour
df['weekday'] = df['event_time'].dt.day_name()
df['month'] = df['event_time'].dt.month_name()

#----------
funnel_counts = df['event_type'].value_counts()

views = funnel_counts['view']
carts = funnel_counts['cart']
checkouts = funnel_counts['checkout']
purchases = funnel_counts['purchase']

cart_rate = carts / views
checkout_rate = checkouts / carts
purchase_rate = purchases / checkouts

funnel_summary = pd.DataFrame({
    'stage' : ['view', 'cart', 'checkout','purchase'],
    'count' : [views, carts, checkouts, purchases],
    'conversion_from_previous': [np.nan, cart_rate, checkout_rate, purchase_rate]
})

print ("\n GLOBAL FUNNEL SUMMARY ")
print (funnel_summary)

#--------
category_funnel = (
    df.groupby(['category', 'event_type'])
        .size()
        .unstack()
        .fillna(0)
        .astype(int)
)

category_funnel['view_to_cart_rate'] = category_funnel['cart'] / category_funnel['view']
category_funnel['cart_to_checkout_rate'] = category_funnel['checkout'] / category_funnel['cart']
category_funnel['checkout_to_purchase_rate'] = category_funnel['purchase'] / category_funnel['checkout']

print("\n CATEGORY FUNNEL ")
print(category_funnel)

#-------
device_funnel = (
    df.groupby(['device', 'event_type'])
        .size()
        .unstack()
        .fillna(0)
        .astype(int)
)

device_funnel['view_to_cart_rate'] = device_funnel['cart'] / device_funnel['view']
device_funnel['cart_to_checkout_rate'] = device_funnel['checkout'] / device_funnel['cart']
device_funnel['checkout_to_purchase_rate'] = device_funnel['purchase'] / device_funnel['checkout']    

print("\n DEVICE FUNNEL ")
print(device_funnel)

#--------
daily_events = (
    df.groupby(['date', 'event_type'])
        .size()
        .unstack()
        .fillna(0)
        .astype(int)
)

hourly_events = (
    df.groupby(['hour', 'event_type'])
        .size()
        .unstack()
        .fillna(0)
        .astype(int)
)

print("\n DAILY EVENTS ")
print(daily_events.head())

print("\n HOURLY EVENTS")
print(hourly_events.head())

#-------

os.makedirs('visuals', exist_ok=True)
#-------
plt.figure(figsize=(6,4))
sns.barplot(x=funnel_counts.index, y=funnel_counts.values, hue=funnel_counts.index, palette="Blues_d", legend=False)
plt.title("Ecommerce funnel event counts")
plt.xlabel("Event type")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig('visuals/funnel_counts.png')
plt.close()
#-------
plt.figure(figsize=(10,6))
sns.heatmap(category_funnel[['view', 'cart', 'checkout', 'purchase']], annot=True, fmt='g', cmap="Blues")
plt.title("Category Funnel performance")
plt.xlabel("Event Type")
plt.ylabel("Category")
plt.tight_layout()
plt.savefig('visuals/category_funnel.png')
plt.close()
#-------
plt.figure(figsize=(6,4))
sns.heatmap(device_funnel[['view', 'cart', 'checkout', 'purchase']], annot=True, fmt='g', cmap="Greens")
plt.title("Device Funnel Performance")
plt.xlabel("Event Type")
plt.ylabel("Device")
plt.tight_layout()
plt.savefig('visuals/device_funnel.png')
plt.close()
#--------
plt.figure(figsize=(10,6))
hourly_events.plot(ax=plt.gca())
plt.title("Hourly events trends")
plt.xlabel("Hour of Day")
plt.ylabel("Event Count")
plt.tight_layout()
plt.savefig('visuals/hourly_events.png')
plt.close()

print("\n All visuals saved to /visuals folder.")
print("Analysis completed successfully")



 GLOBAL FUNNEL SUMMARY 
      stage  count  conversion_from_previous
0      view  30009                       NaN
1      cart   9974                  0.332367
2  checkout   5005                  0.501805
3  purchase   5012                  1.001399

 CATEGORY FUNNEL 
event_type   cart  checkout  purchase  view  view_to_cart_rate  \
category                                                         
appliances   1010       506       496  2882           0.350451   
audio         993       461       500  3005           0.330449   
cameras       981       508       495  2942           0.333447   
gaming        997       498       472  2991           0.333333   
laptops      1005       523       488  2997           0.335335   
monitors     1024       481       513  3035           0.337397   
networking    988       540       526  2973           0.332324   
smart_home    996       498       526  3078           0.323587   
smartphones   960       504       520  3095           0.310178   
table